In [ ]:
from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.datastructures.prefixed_name import PrefixedName
from semantic_digital_twin.robots.abstract_robot import AbstractRobot
from pycram.datastructures.dataclasses import Context
from semantic_digital_twin.semantic_annotations.semantic_annotations import Milk
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Vector3
from semantic_digital_twin.world_description.connections import (
    OmniDrive,
    DiffDrive,
    FixedConnection,
    Connection6DoF,
    RevoluteConnection,
)
from semantic_digital_twin.world_description.world_entity import (
    Body,
)
import os

from typing_extensions import Type
from semantic_digital_twin.adapters.package_resolver import PathResolver
from copy import deepcopy

import logging
import pathlib
from dotenv import load_dotenv

from pycram.datastructures.grasp import GraspDescription
# ── SDT / pycram stack ────────────────────────────────────────────────────────
from semantic_digital_twin.robots.pr2 import PR2
from pycram.datastructures.dataclasses import Context

# ── Generative backend — pipeline entry point ─────────────────────────────────
from generative_backend.pipeline import ActionPipeline

# ── Schemas (common + action-specific) ───────────────────────────────────────
from generative_backend.workflows.schemas.common import EntityDescriptionSchema
from generative_backend.workflows.schemas.pick_up import PickUpSlotSchema
from generative_backend.workflows.schemas.place import PlaceSlotSchema

# ── LangGraph nodes (Phase 1 + generic Phase 2) ───────────────────────────────
from generative_backend.workflows.nodes.slot_filler import run_slot_filler
from generative_backend.workflows.nodes.resolver import (
    run_resolver,
    run_pickup_resolver,
    run_place_resolver,
)
logging.getLogger("generative_backend").setLevel(logging.DEBUG)
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
print("All imports OK")

In [ ]:
# Load API keys before any LLM import
_env = pathlib.Path().resolve().parent / ".env"
load_dotenv(_env, override=True)

def apartment_world_setup():
    apartment_world = URDFParser.from_file(
        os.path.join(
            os.path.dirname(os.getcwd()),
            "..",
            "pycram",
            "resources",
            "worlds",
            "apartment.urdf",
        )
    ).parse()
    milk_world = STLParser(
        os.path.join(
            os.path.dirname(os.getcwd()),
            "..",
            "pycram",
            "resources",
            "objects",
            "milk.stl",
        )
    ).parse()
    cereal_world = STLParser(
        os.path.join(
            os.path.dirname(os.getcwd()),
            "..",
            "pycram",
            "resources",
            "objects",
            "breakfast_cereal.stl",
        )
    ).parse()
    apartment_world.merge_world_at_pose(
        milk_world,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 2, 1.05, reference_frame=apartment_world.root
        ),
    )
    apartment_world.merge_world_at_pose(
        cereal_world,
        HomogeneousTransformationMatrix.from_xyz_rpy(
            2.37, 1.8, 1.05, reference_frame=apartment_world.root
        ),
    )
    milk_view = Milk(
        root=apartment_world.get_body_by_name("milk.stl"), _world=apartment_world
    )
    with apartment_world.modify_world():
        apartment_world.add_semantic_annotation(milk_view)

    return apartment_world

def world_with_urdf_factory(
    urdf_path: str,
    robot_semantic_annotation: Type[AbstractRobot] | None,
    drive_connection_type: Type[OmniDrive | DiffDrive],
    robot_starting_pose: HomogeneousTransformationMatrix | None = None,
    urdf_path_resolver: PathResolver | None = None,
):
    """
    Builds this tree:
    map -> odom_combined -> "urdf tree"
    """
    urdf_parser = URDFParser.from_file(
        file_path=urdf_path, path_resolver=urdf_path_resolver
    )
    world_with_urdf = urdf_parser.parse()
    if robot_semantic_annotation is not None:
        robot_semantic_annotation.from_world(world_with_urdf)

    with world_with_urdf.modify_world():
        map = Body(name=PrefixedName("map"))
        localization_body = Body(name=PrefixedName("odom_combined"))

        map_C_localization = Connection6DoF.create_with_dofs(
            world_with_urdf, map, localization_body
        )
        world_with_urdf.add_connection(map_C_localization)

        c_root_bf = drive_connection_type.create_with_dofs(
            parent=localization_body,
            child=world_with_urdf.root,
            world=world_with_urdf,
        )
        world_with_urdf.add_connection(c_root_bf)
        c_root_bf.has_hardware_interface = True
        if robot_starting_pose is not None:
            c_root_bf.origin = robot_starting_pose

    return world_with_urdf

def pr2_world_setup():
    urdf_dir = "package://iai_pr2_description/robots/pr2_with_ft2_cableguide.xacro"
    return world_with_urdf_factory(urdf_dir, PR2, OmniDrive)

def pr2_apartment_world(pr2_world_setup, apartment_world_setup):
    """
    Builds this tree:
    map -> odom_combined -> pr2 urdf tree
        -> apartment urdf
    """
    pr2_copy = deepcopy(pr2_world_setup)
    PR2.from_world(pr2_copy)  # semantic annotations are lost on copy

    apartment_copy = deepcopy(apartment_world_setup)

    pr2_copy.merge_world(apartment_copy)
    pr2_copy.get_body_by_name("base_footprint").parent_connection.origin = (
        HomogeneousTransformationMatrix.from_xyz_rpy(1.3, 2, 0)
    )
    return pr2_copy

def mutable_model_world(pr2_apartment_world):
    world = deepcopy(pr2_apartment_world)
    pr2 = PR2.from_world(world)
    return world, pr2, Context(world, pr2)

In [ ]:
world, pr2, context = mutable_model_world(pr2_apartment_world(pr2_world_setup(), apartment_world_setup()))

In [ ]:
# world.get_body_by_name("milk.stl")._semantic_annotations

In [ ]:
# set([type(sem) for sem in world.semantic_annotations])

In [ ]:
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
import threading
import rclpy

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [ ]:
_tf_publisher  = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print('ROS2 publishers started')
print('  Fixed Frame : apartment/apartment_root')
print('  TF topic    : /tf')
print('  Marker topic: /semworld/viz_marker  (QoS Durability=Transient Local)')

In [ ]:
pipeline = ActionPipeline.from_world(world)

print("ActionPipeline ready")
print(f"  manipulator : {type(pipeline.world_context.manipulator).__name__}")

# Show which action types are registered in the dispatcher
from generative_backend.pipeline import ActionDispatcher
from pycram.robot_plans import (
    ParkArmsActionDescription,
    MoveTorsoActionDescription,
    NavigateActionDescription,
    PickUpActionDescription,
    PlaceActionDescription,
)
from pycram.datastructures.grasp import GraspDescription
from pycram.language import SequentialPlan
from pycram.motion_executor import simulated_robot
from pycram.designators.location_designator import CostmapLocation, SemanticCostmapLocation
from pycram.datastructures.enums import ApproachDirection, VerticalAlignment, Arms
from pycram.datastructures.pose import PoseStamped
from semantic_digital_twin.datastructures.definitions import TorsoState
print(f"  registered action types: {list(ActionDispatcher._registry.keys())}")

In [ ]:
instructions = [
    "Pick up the milk from the counter",
    "Place the milk on the table_area_main",
]
plans = []
for instr in instructions:
    action = pipeline.run(instr)
    plans.append(action)
    print(f"Instruction : {instr!r}")
    print(f"  → {type(action).__name__}")
    print(f"     arm = {action.arm}")
    if hasattr(action, "grasp_description") and action.grasp_description:
        gd = action.grasp_description
        print(f"     approach_direction = {gd.approach_direction}")
        print(f"     vertical_alignment = {gd.vertical_alignment}")
        print(f"     rotate_gripper     = {gd.rotate_gripper}")
    if hasattr(action, "target_location") and action.target_location is not None:
        tgt = str(getattr(getattr(action.target_location, "name", None), "name",
                          action.target_location))
        print(f"     target_location    = {tgt}")
    print()

In [ ]:
# plans[1].__dict__
# place_pose = PoseStamped.from_list([5.7, 4.38, 1.0], [0, 0, 0.969, -0.246], world.root)
# dplace_pose = PoseStamped.from_spatial_type(plans[-1].target_location.global_pose.to_pose())

# grasp_descs = GraspDescription.calculate_grasp_descriptions(plans[0].grasp_description.manipulator, PoseStamped.from_spatial_type(plans[0].object_designator.global_pose))

# sem_place_nav_loc = SemanticCostmapLocation(body=place_body, for_object=pick_body)
# place_nav_loc = CostmapLocation(target=place_pose, reachable_for=pr2, reachable_arm=Arms.RIGHT)
# pickad = PickUpActionDescription(object_designator=plans[0].object_designator,arm=plans[0].arm,grasp_description=gd)
# placead = PlaceActionDescription(object_designator=plans[1].object_designator,target_location=dynamic_place_pose, arm=Arms.RIGHT)

In [ ]:
pick_body = plans[0].object_designator
pick_body_pose = PoseStamped.from_spatial_type(pick_body.global_pose)
place_body = plans[1].target_location
place_body_pose = PoseStamped.from_spatial_type(place_body.global_pose)
table_surface = world.get_body_by_name("island_countertop")

In [ ]:
print(pick_body)
print(pick_body_pose)
print(place_body)
print(place_body_pose)

In [ ]:
gd = GraspDescription(approach_direction=plans[0].grasp_description.approach_direction, vertical_alignment=VerticalAlignment.NoAlignment, rotate_gripper=plans[0].grasp_description.rotate_gripper, manipulator=plans[0].grasp_description.manipulator)
pickup_arm = plans[0].arm

pick_nav_loc = CostmapLocation(target=pick_body_pose, reachable_for=pr2, reachable_arm=pickup_arm)
# Dynamic placement pose: surface of table + milk height offset
sem_place_loc = SemanticCostmapLocation(body=table_surface, for_object=pick_body)
dynamic_place_pose = sem_place_loc.ground()
# Dynamic navigation pose: where robot can stand to reach that spot
place_nav_loc = CostmapLocation(target=dynamic_place_pose, reachable_for=pr2, reachable_arm=pickup_arm)

In [ ]:
candidates = ["island_countertop", "countertop", "coffee_table", "table_area_main"]
for name in candidates:
  try:
      b = world.get_body_by_name(name)
      print(f"{name}: {b.global_pose.to_position()}")
  except:
      print(f"{name}: not found")

In [ ]:
with simulated_robot:
    SequentialPlan(
        context,
        ParkArmsActionDescription(Arms.BOTH),
        MoveTorsoActionDescription([TorsoState.HIGH]),
        NavigateActionDescription(pick_nav_loc),
        PickUpActionDescription(
            object_designator=pick_body,
            arm=[pickup_arm],
            grasp_description=gd,
        ),
        # MoveTorsoActionDescription([TorsoState.LOW]),
        NavigateActionDescription(place_nav_loc),
        PlaceActionDescription(
            object_designator=pick_body,
            target_location=dynamic_place_pose,
            arm=pickup_arm,
        ),
    ).perform()

In [ ]:
place_pose = PoseStamped.from_list([5.0, -5.0, 1.0], [0, 0, 0.707, 0.707], world.root)

In [ ]:
with simulated_robot:
    SequentialPlan(context,
        NavigateActionDescription(place_pose),
    ).perform()

In [ ]:
sp = SequentialPlan(context,NavigateActionDescription(CostmapLocation(target=pick_body_pose, reachable_for=pr2, reachable_arm=Arms.RIGHT)))

In [ ]:
sp.nodes[1].__dict__.keys()

In [ ]:
sp.nodes[1].__dict__['designator_type'].__dict__